In [1]:
import cv2
import numpy as np
import tensorflow as tf
import os

In [2]:
from keras import models
from keras import layers
from keras import optimizers
from keras.preprocessing.image import ImageDataGenerator

In [3]:
Imagesize=64
model = models.Sequential()
model.add(layers.Conv2D(32, (3, 3), activation='relu', input_shape=(Imagesize,Imagesize,3)))
model.add(layers.MaxPooling2D((2, 2)))
model.add(layers.Conv2D(64, (3, 3), activation='relu'))
model.add(layers.MaxPooling2D(2, 2))
model.add(layers.Conv2D(128, (3, 3), activation='relu'))
model.add(layers.MaxPooling2D(2, 2))
model.add(layers.Conv2D(256, (3, 3), activation='relu'))
model.add(layers.MaxPooling2D(2, 2))
model.add(layers.Flatten())
model.add(layers.Dropout(0.25))
model.add(layers.Dense(512, activation='relu'))
model.add(layers.Dense(25, activation='softmax'))

In [6]:
import keras

In [7]:
model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
reduce_lr = keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=1, min_lr=0.0001)
early_stop = keras.callbacks.EarlyStopping(monitor='val_loss', min_delta=0, patience=2, verbose=0, mode='auto')

In [8]:
model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 conv2d (Conv2D)             (None, 62, 62, 32)        896       
                                                                 
 max_pooling2d (MaxPooling2  (None, 31, 31, 32)        0         
 D)                                                              
                                                                 
 conv2d_1 (Conv2D)           (None, 29, 29, 64)        18496     
                                                                 
 max_pooling2d_1 (MaxPoolin  (None, 14, 14, 64)        0         
 g2D)                                                            
                                                                 
 conv2d_2 (Conv2D)           (None, 12, 12, 128)       73856     
                                                                 
 max_pooling2d_2 (MaxPoolin  (None, 6, 6, 128)         0

In [9]:
data_train = ImageDataGenerator(rescale=1.0/255.0,rotation_range=40,width_shift_range=0.2,height_shift_range=0.2,
                                shear_range=0.2,zoom_range=0.2,horizontal_flip=True)
data_valid = ImageDataGenerator(rescale=1.0/255.0)

In [10]:
batch = 96

In [12]:
train_gen = data_train.flow_from_directory('train_processed',target_size=(Imagesize,Imagesize),batch_size=batch, 
                                           class_mode='categorical')

test_gen = data_valid.flow_from_directory('test_processed', target_size=(Imagesize,Imagesize), batch_size=batch, 
                                           class_mode='categorical')

Found 8000 images belonging to 25 classes.
Found 2002 images belonging to 25 classes.


In [13]:
model.fit_generator(train_gen, epochs=3, steps_per_epoch=50, validation_data = test_gen, 
                    validation_steps=4, callbacks=[reduce_lr, early_stop])

C:\Users\Asim\AppData\Local\Temp\ipykernel_37344\140172344.py:1: UserWarning: `Model.fit_generator` is deprecated and will be removed in a future version. Please use `Model.fit`, which supports generators.
  model.fit_generator(train_gen, epochs=3, steps_per_epoch=50, validation_data = test_gen,


Epoch 1/3


50/50 [==============================] - 4s 73ms/step - loss: 2.9233 - accuracy: 0.1269 - val_loss: 1.6778 - val_accuracy: 0.5885 - lr: 0.0010
Epoch 2/3
50/50 [==============================] - 4s 73ms/step - loss: 1.9938 - accuracy: 0.3761 - val_loss: 0.5912 - val_accuracy: 0.8438 - lr: 0.0010
Epoch 3/3
50/50 [==============================] - 4s 73ms/step - loss: 1.2070 - accuracy: 0.6058 - val_loss: 0.1293 - val_accuracy: 0.9818 - lr: 0.0010


In [14]:
#to create a saved model, use the following command
model.save('islcnnmodel1.h5')

c:\Users\Asim\anaconda3\envs\grayscale_to_rgb\Lib\site-packages\keras\src\engine\training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


In [15]:
imgs,labels=next(test_gen)
scores = model.evaluate(imgs,labels,verbose=0)
print("Accuracy: ",scores[1])

Accuracy:  1.0
